# NBA Injury-Risk — Stage 0–4 Walkthrough

Reproducible path from raw pulls to the discrete-time hazard model.

**Prerequisite:** run the network-dependent pulls first on a machine with open egress:
```bash
python -m nba_injury.gate0_hello
python -m nba_injury.fetch_injuries --start 2015-10-01 --end 2025-07-01
python -m nba_injury.build_labels            # Gate 1
python -m nba_injury.rosters
python -m nba_injury.pull_features           # overnight; resumable
python -m nba_injury.fetch_bbref --start 2016 --end 2025
python -m nba_injury.gate2_report            # Gate 2
python -m nba_injury.build_person_period     # Gate 3 input
python -m nba_injury.gate3_report            # Gate 3
```

Without network access, generate a **synthetic** world to exercise the
pipeline (clearly not real data):
```bash
python -m nba_injury.make_synthetic_world
python -m nba_injury.build_person_period
```

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / 'src'))
import pandas as pd
from nba_injury.cache import processed_path

## 1. Load the canonical person-period table (Stage 3 output)
One row = one player-week, active weeks only, strictly causal features.

In [ ]:
df = pd.read_parquet(processed_path('person_period.parquet'))
print(df.shape)
print('players:', df.player_id.nunique(), '| events:', int(df.event.sum()))
df.head()

## 2. Class imbalance is the central challenge
Weekly injury onset is rare — this is why we report AUC-PR + calibration,
**not accuracy**.

In [ ]:
at_risk = df[df.at_risk == 1]
rate = at_risk.event.mean()
print(f'at-risk player-weeks: {len(at_risk)}')
print(f'weekly injury-onset prevalence: {rate:.4f}  (~1 in {1/rate:.0f})')

## 3. Strict temporal validation + the two models
Train on earlier seasons, test on later — never a random split.

In [ ]:
from nba_injury.model_hazard import run
results = run()  # prints the metrics table

## 4. Calibration
Are the predicted hazards trustworthy probabilities, or just rankings?

In [ ]:
import matplotlib.pyplot as plt
cal = results['calibration']['hgb_tier1_2']
if cal:
    xs = [p['mean_pred'] for p in cal]; ys = [p['frac_pos'] for p in cal]
    plt.plot([0, max(xs)], [0, max(xs)], '--', label='perfect')
    plt.plot(xs, ys, 'o-', label='HGB')
    plt.xlabel('mean predicted hazard'); plt.ylabel('observed frequency')
    plt.legend(); plt.title('Calibration (test seasons)'); plt.show()
else:
    print('Not enough events in the test split for a calibration curve.')

## 5. Gate 4
The model must beat both the constant league-average-hazard predictor and the
age+minutes-only baseline, or we stop and investigate before adding depth.

In [ ]:
from nba_injury.gate4_report import main as gate4
gate4()

---
**Next:** Stage 5 — the hindsight-bias / validation gauntlet (leakage audit,
stylistic-comparables check on the Rose/Zion cases against healthy comparables).